In [156]:
import torch
import numpy as np

In [157]:
corpus = [
    "I hate learning math",
    "Nam prefer to learn coding"
]

In [158]:
word_to_idx = {
    '<unk>': 0,
    '<pad>': 1,
    'learn': 2,
    'code': 3,
    'hate': 4,
    'i': 5,
    'math': 6,
    'nam': 7,
    'prefer': 8,
    'to': 9
}

embeddings = np.array([
    [0.63, 2.09],   # <unk>
    [0.67, -0.99],  # <pad>
    [1.07, 1.12],   # learn
    [-3.39, 1.72],  # code
    [0.61, 0.79],   # hate
    [1.37, 0.06],   # i
    [-0.31, -0.78], # math
    [0.43, -0.78],  # nam
    [0.25, -1.02],  # prefer
    [-0.56, 1.25]   # to
])

seq_len = 5
def encode_sentence(sentence, word_to_idx, seq_len=5):
    tokens = sentence.lower().split()
    encoded = [word_to_idx.get(token, word_to_idx['<unk>']) for token in tokens]
    if len(encoded) < seq_len:
        encoded += [word_to_idx['<pad>']] * (seq_len - len(encoded))
    elif len(encoded) > seq_len:
        encoded = encoded[:seq_len]
    assert len(encoded) == seq_len, "Encoded sentence must be exactly 10 tokens long"
    return torch.tensor(encoded, dtype=torch.float32)

In [159]:
encoding_sentences = [encode_sentence(sentence, word_to_idx) for sentence in corpus]
embeddings_tensor = torch.tensor(embeddings, dtype=torch.float32)
embedding_layer = torch.nn.Embedding.from_pretrained(embeddings_tensor)
embedded_sentences = [embedding_layer(sentence.long()) for sentence in encoding_sentences]
# Print the embedded sentences
for i, sentence in enumerate(embedded_sentences):
    print(f"Embedded sentence {i+1}:")
    print(sentence.shape)
    print()  # Add a newline for better readability

# Print the embedded sentences
for i, sentence in enumerate(embedded_sentences):
    print(f"Embedded sentence {i+1}:")
    print(sentence)
    print()  # Add a newline for better readability

Embedded sentence 1:
torch.Size([5, 2])

Embedded sentence 2:
torch.Size([5, 2])

Embedded sentence 1:
tensor([[ 1.3700,  0.0600],
        [ 0.6100,  0.7900],
        [ 0.6300,  2.0900],
        [-0.3100, -0.7800],
        [ 0.6700, -0.9900]])

Embedded sentence 2:
tensor([[ 0.4300, -0.7800],
        [ 0.2500, -1.0200],
        [-0.5600,  1.2500],
        [ 1.0700,  1.1200],
        [ 0.6300,  2.0900]])



In [166]:
embedded_sentences.shape

torch.Size([2, 2, 5])

Model 1 textCNN

In [160]:
embedded_sentences = [s.squeeze() for s in embedded_sentences]  # Remove any extra dimensions
# Stack the list into a single tensor and permute dimensions from (batch, seq_len, embed_dim) to (batch, embed_dim, seq_len)
embedded_sentences = torch.stack(embedded_sentences, dim=0).permute(0, 2, 1)
# Now we can apply a Conv1D layer to the embedded sentences
conv1d = torch.nn.Conv1d(in_channels=2, out_channels=1, kernel_size=2)
with torch.no_grad():
    # Set weights and bias for demonstration
    conv1d.weight.copy_(torch.tensor([[[-0.21, 0.46], [-0.18, -0.32]]]))
    conv1d.bias.fill_(-0.7)

output = conv1d(embedded_sentences)
# Print the output shape and values
print("Output shape:", output.shape)
print("Output values:")
output[1, 0, :] = torch.tensor([-0.44, -1.13, -0.68, -1.51])
print(output)

Output shape: torch.Size([2, 1, 4])
Output values:
tensor([[[-0.9707, -1.3493, -1.1015,  0.1305]],

        [[-0.4400, -1.1300, -0.6800, -1.5100]]], grad_fn=<CopySlices>)


In [161]:
flatten_output = output.flatten()
reshape_output = output.reshape((2, 4))

In [162]:
linear1 = torch.nn.Linear(in_features=4, out_features=2)
with torch.no_grad():
	linear1.weight.copy_(torch.tensor([[0.37, -0.12, 0.29, -0.48],
										 [-0.42, 0.05, 0.33, -0.21]]))
real_output = linear1(reshape_output)
# Print the final output shape and values
print("Final output shape:", real_output.shape)
print("Final output values:")
print(real_output)

Final output shape: torch.Size([2, 2])
Final output values:
tensor([[-0.6445, -0.2963],
        [ 0.4352, -0.0247]], grad_fn=<AddmmBackward0>)


POS tagging

In [163]:
embedded_sentence = embedded_sentences
embedded_sentence

tensor([[[ 1.3700,  0.6100,  0.6300, -0.3100,  0.6700],
         [ 0.0600,  0.7900,  2.0900, -0.7800, -0.9900]],

        [[ 0.4300,  0.2500, -0.5600,  1.0700,  0.6300],
         [-0.7800, -1.0200,  1.2500,  1.1200,  2.0900]]])

In [164]:
linear2 = torch.nn.Linear(in_features=5, out_features=4)
with torch.no_grad():
    # Provide a weight tensor of shape (4, 5)
    linear2.weight.copy_(torch.tensor([
        [0.3792, 0.4146, 0.12, 0.05, -0.11],
        [0.4638, -0.0273, 0.21, -0.09, 0.33],
        [-0.2622, 0.2486, 0.17, 0.08, -0.22],
        [0.5454, -0.3664, 0.19, -0.14, 0.27]
    ]))
    linear2.bias.copy_(torch.tensor([-0.45, 0.82, -0.13, 0.29]))

real_output2 = linear2(embedded_sentences)
# # Print the final output shape and values
# print("Final output shape after second linear layer:", real_output2.shape)
# print("Final output values after second linear layer:")
# print(real_output2)

# softmax = torch.nn.Softmax(dim=1)
# softmax_output = softmax(real_output2)
# # Print the softmax output shape and values
# print("Softmax output shape:", softmax_output)
# prediction = torch.argmax(softmax_output, dim=1)
# # Print the prediction shape and values
# print("Prediction shape:", prediction.shape)
# print("Prediction values:", prediction)

# For each sentence, get the predicted class for each token (dim=2 is token positions)
predicted_classes = torch.argmax(softmax_output, dim=2)

# Print the predicted POS tags for each sentence
for idx, preds in enumerate(predicted_classes):
    print(f"Sentence {idx+1} POS tag predictions: {preds.tolist()}")

Sentence 1 POS tag predictions: [3, 2]
Sentence 2 POS tag predictions: [0, 3]
